In [ ]:
LAYER = 0  # Zero-based transformer block whose post-residual stream is probed.

print("here")

from pathlib import Path
import json

print("here")

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import torch
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

print("here")

import project_main.checkpoints as project_checkpoints
import project_main.data as project_data
import project_main.model as project_model
import project_main.tokens as project_tokens

print("here")

RUN_DIR = Path("../results/line_breaks_results")
CHECKPOINT_PATH = RUN_DIR / "checkpoints" / "final.pt"
N_TRAIN_SEQUENCES = 2_000
N_EVAL_SEQUENCES = 5_000
COLLECTION_BATCH_SIZE = 128
TRAIN_SEED = 17_290
EVAL_SEED = 17_291
ANNOTATE_EVERY = 5
PLOT_HEIGHT = 760
print(f"Layer: {LAYER}")

# Joint geometry of character-count and line-width probes

This notebook trains two multinomial logistic classifiers on the residual stream immediately after zero-based block `LAYER`:

- character-count probes, matching Notebook 6; and
- line-width probes predicting the true sampled line width, trained only at positions where the true next-token target is `_NEWLINE_`.

The first 3-D plot jointly fits PCA to both probe families for values 50 through 100 and connects the two probes with the same value. The later plots repeat that joint-PCA construction for every head in the next block: character-count probes are multiplied by that head's query matrix and line-width probes by its key matrix. Because `blocks.{LAYER}.hook_resid_post` is the input stream to block `LAYER + 1`, those are the downstream heads that directly consume the probed representation.

The notebook uses Plotly for the 3-D figures. Drag directly inside a plot to rotate it, scroll to zoom, and double-click to reset the camera. The camera controls remain interactive without Matplotlib's widget backend.


In [ ]:
with open(RUN_DIR / "config.json", "r") as f:
    cfg = json.load(f)

min_line_width, max_line_width = project_data.get_line_length_range(cfg["task"])
if (min_line_width, max_line_width) != (50, 100):
    raise ValueError(
        "This analysis expects configured line widths 50 through 100, but got "
        f"{min_line_width} through {max_line_width}."
    )

n_layers = cfg["model"]["n_layers"]
if not 0 <= LAYER < n_layers - 1:
    raise ValueError(
        f"LAYER must be between 0 and {n_layers - 2} so a downstream "
        "attention layer exists."
    )

PROBE_ACTIVATION_SITE = f"blocks.{LAYER}.hook_resid_post"
ATTENTION_LAYER = LAYER + 1
LINE_WIDTH_VALUES = np.arange(50, 101)
_, configured_max_line_width = project_data.get_line_length_range(cfg["task"])
max_character_count = (
    configured_max_line_width + cfg["task"]["num_token_lengths"] - 1
)
CHARACTER_COUNT_VALUES = np.arange(1, max_character_count + 1)

device = "cuda" if torch.cuda.is_available() else "cpu"
vocab = project_tokens.build_vocab(cfg["task"])
model = project_model.build_model(cfg=cfg, device=device)
checkpoint = project_checkpoints.load_checkpoint(
    path=CHECKPOINT_PATH,
    model=model,
    map_location=device,
)
model.eval()

print(f"Loaded checkpoint step {checkpoint['step']} on {device}")
print(f"Probe site: {PROBE_ACTIVATION_SITE}")
print(f"Downstream query/key matrices: attention layer {ATTENTION_LAYER}")
print(f"Layer: {LAYER}")

## Collect the two probe datasets

Character counts are aligned with input-token residual positions, with count-zero BOS and newline positions removed as in Notebook 6. The line-width dataset is separate: it retains a residual vector only when that position's true next-token target is `_NEWLINE_`, and labels it with the example's sampled line width. Thus the line-width probe is trained and evaluated exclusively at positions where the model should predict a newline.


In [ ]:
def character_counts_for_tokens(token_rows: torch.Tensor) -> torch.Tensor:
    """Return the running character count at every token position."""
    all_counts = []

    for token_row in token_rows.detach().cpu().tolist():
        running_count = 0
        row_counts = []
        for token_id in token_row:
            if token_id == vocab.newline_id:
                running_count = 0
            else:
                running_count += project_data.extract_character_count(
                    vocab.decode_token(token_id)
                )
            row_counts.append(running_count)
        all_counts.append(row_counts)

    return torch.tensor(all_counts, dtype=torch.long)


@torch.no_grad()
def collect_probe_dataset(
    n_sequences: int,
    *,
    seed: int,
    batch_size: int = COLLECTION_BATCH_SIZE,
) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    character_activation_chunks = []
    character_count_chunks = []
    line_width_activation_chunks = []
    line_width_chunks = []

    for start in range(0, n_sequences, batch_size):
        this_batch_size = min(batch_size, n_sequences - start)
        batch = project_data.make_batch(
            batch_size=this_batch_size,
            vocab=vocab,
            task_cfg=cfg["task"],
            device=device,
            seed=seed + start,
        )
        counts = character_counts_for_tokens(batch.tokens)
        line_widths = (
            batch.line_lengths.detach().cpu()[:, None].expand_as(counts)
        )
        saved = {}

        def save_residuals(activation, hook):
            saved["residuals"] = activation.detach().cpu()

        model.run_with_hooks(
            batch.tokens,
            prepend_bos=False,
            fwd_hooks=[(PROBE_ACTIVATION_SITE, save_residuals)],
        )

        character_keep = counts > 0
        newline_target = batch.targets.detach().cpu() == vocab.newline_id
        residuals = saved["residuals"]

        character_activation_chunks.append(
            residuals[character_keep].float().numpy()
        )
        character_count_chunks.append(counts[character_keep].numpy())
        line_width_activation_chunks.append(
            residuals[newline_target].float().numpy()
        )
        line_width_chunks.append(line_widths[newline_target].numpy())

    return (
        np.concatenate(character_activation_chunks),
        np.concatenate(character_count_chunks),
        np.concatenate(line_width_activation_chunks),
        np.concatenate(line_width_chunks),
    )
print(f"Layer: {LAYER}")

In [ ]:
(
    X_character_train,
    y_character_train,
    X_width_train,
    y_width_train,
) = collect_probe_dataset(
    N_TRAIN_SEQUENCES, seed=TRAIN_SEED
)
(
    X_character_eval,
    y_character_eval,
    X_width_eval,
    y_width_eval,
) = collect_probe_dataset(
    N_EVAL_SEQUENCES, seed=EVAL_SEED
)

for name, expected, observed in [
    ("character count", CHARACTER_COUNT_VALUES, y_character_train),
    ("line width", LINE_WIDTH_VALUES, y_width_train),
]:
    missing = sorted(set(expected) - set(np.unique(observed)))
    if missing:
        raise RuntimeError(
            f"Training data is missing {name} classes {missing}. "
            "Increase N_TRAIN_SEQUENCES."
        )

print("Character train residuals:", X_character_train.shape)
print("Character eval residuals: ", X_character_eval.shape)
print("Newline-target width train residuals:", X_width_train.shape)
print("Newline-target width eval residuals: ", X_width_eval.shape)
print("Character-count range:", y_character_train.min(), "to", y_character_train.max())
print("Line-width range:", y_width_train.min(), "to", y_width_train.max())
print(f"Layer: {LAYER}")


## Train both multinomial probe families

Each classifier has one learned logit direction per class. As in Notebook 6, features are standardized before fitting. Coefficients are then divided by the fitted feature scales to return them to residual-stream coordinates. Finally, each probe family is mean-centered to remove multinomial logistic regression's shared-vector degree of freedom.


In [ ]:
def fit_probe(X: np.ndarray, y: np.ndarray):
    probe = make_pipeline(
        StandardScaler(),
        LogisticRegression(
            C=1.0,
            solver="lbfgs",
            max_iter=1_000,
            tol=1e-3,
        ),
    )
    return probe.fit(X, y)


character_probe_model = fit_probe(X_character_train, y_character_train)
line_width_probe_model = fit_probe(X_width_train, y_width_train)

for name, probe, X_task_eval, y_task_eval in [
    ("Character count", character_probe_model, X_character_eval, y_character_eval),
    ("Line width at newline targets", line_width_probe_model, X_width_eval, y_width_eval),
]:
    predictions = probe.predict(X_task_eval)
    print(
        f"{name} exact accuracy: "
        f"{accuracy_score(y_task_eval, predictions):.2%}"
    )
    print(f"{name} MAE: {np.mean(np.abs(predictions - y_task_eval)):.3f}")


def residual_coordinate_probe_vectors(probe_model):
    scaler = probe_model.named_steps["standardscaler"]
    classifier = probe_model.named_steps["logisticregression"]
    vectors = classifier.coef_ / scaler.scale_[None, :]
    vectors -= vectors.mean(axis=0, keepdims=True)
    return classifier.classes_, vectors


character_classes, character_probe_vectors = residual_coordinate_probe_vectors(
    character_probe_model
)
width_classes, line_width_probe_vectors = residual_coordinate_probe_vectors(
    line_width_probe_model
)

if not np.array_equal(character_classes, CHARACTER_COUNT_VALUES):
    raise RuntimeError("Character-count probe classes are not the expected values.")
if not np.array_equal(width_classes, LINE_WIDTH_VALUES):
    raise RuntimeError("Line-width probe classes are not 50 through 100.")

character_indices = np.searchsorted(character_classes, LINE_WIDTH_VALUES)
character_probe_vectors_50_100 = character_probe_vectors[character_indices]

print("Character probe vectors:", character_probe_vectors.shape)
print("Line-width probe vectors:", line_width_probe_vectors.shape)
print(f"Layer: {LAYER}")

## Joint 3-D PCA in residual-stream coordinates

PCA is fit once to the combined 102-vector matrix: 51 character-count directions and 51 line-width directions. Every gray segment connects the character-count and line-width probes for the same numeric value.


In [ ]:
def joint_pca_coordinates(
    character_vectors: np.ndarray,
    width_vectors: np.ndarray,
) -> tuple[PCA, np.ndarray, np.ndarray]:
    if character_vectors.shape != width_vectors.shape:
        raise ValueError("The paired probe-vector arrays must have the same shape.")
    pca = PCA(n_components=3)
    coordinates = pca.fit_transform(
        np.concatenate([character_vectors, width_vectors], axis=0)
    )
    n_values = len(character_vectors)
    return pca, coordinates[:n_values], coordinates[n_values:]


def paired_probe_traces(
    character_coordinates: np.ndarray,
    width_coordinates: np.ndarray,
    *,
    character_name: str,
    width_name: str,
    show_legend: bool,
) -> list[go.Scatter3d]:
    connector_coordinates = [[], [], []]
    for character_point, width_point in zip(character_coordinates, width_coordinates):
        for dimension in range(3):
            connector_coordinates[dimension].extend(
                [character_point[dimension], width_point[dimension], None]
            )

    annotated = LINE_WIDTH_VALUES % ANNOTATE_EVERY == 0
    midpoints = (character_coordinates[annotated] + width_coordinates[annotated]) / 2
    return [
        go.Scatter3d(
            x=connector_coordinates[0],
            y=connector_coordinates[1],
            z=connector_coordinates[2],
            mode="lines",
            line=dict(color="rgba(120,120,120,0.55)", width=2),
            name="Same-value pairing",
            legendgroup="pairing",
            showlegend=show_legend,
            hoverinfo="skip",
        ),
        go.Scatter3d(
            x=character_coordinates[:, 0],
            y=character_coordinates[:, 1],
            z=character_coordinates[:, 2],
            mode="lines+markers",
            line=dict(color="#1f77b4", width=3),
            marker=dict(color="#1f77b4", size=4, symbol="circle"),
            customdata=LINE_WIDTH_VALUES,
            name=character_name,
            legendgroup="character",
            showlegend=show_legend,
            hovertemplate=character_name + " %{customdata}<extra></extra>",
        ),
        go.Scatter3d(
            x=width_coordinates[:, 0],
            y=width_coordinates[:, 1],
            z=width_coordinates[:, 2],
            mode="lines+markers",
            line=dict(color="#ff7f0e", width=3),
            marker=dict(color="#ff7f0e", size=4, symbol="diamond"),
            customdata=LINE_WIDTH_VALUES,
            name=width_name,
            legendgroup="width",
            showlegend=show_legend,
            hovertemplate=width_name + " %{customdata}<extra></extra>",
        ),
        go.Scatter3d(
            x=midpoints[:, 0],
            y=midpoints[:, 1],
            z=midpoints[:, 2],
            mode="text",
            text=LINE_WIDTH_VALUES[annotated].astype(str),
            textfont=dict(size=10, color="#666666"),
            showlegend=False,
            hoverinfo="skip",
        ),
    ]


def pca_scene(pca: PCA) -> dict:
    return dict(
        xaxis_title=f"Joint PC1 ({pca.explained_variance_ratio_[0]:.1%})",
        yaxis_title=f"Joint PC2 ({pca.explained_variance_ratio_[1]:.1%})",
        zaxis_title=f"Joint PC3 ({pca.explained_variance_ratio_[2]:.1%})",
        aspectmode="data",
        camera=dict(eye=dict(x=1.45, y=1.45, z=0.95)),
    )


residual_pca, character_residual_coordinates, width_residual_coordinates = (
    joint_pca_coordinates(
        character_probe_vectors_50_100,
        line_width_probe_vectors,
    )
)

fig = go.Figure()
for trace in paired_probe_traces(
    character_residual_coordinates,
    width_residual_coordinates,
    character_name="Character-count probe",
    width_name="Line-width probe",
    show_legend=True,
):
    fig.add_trace(trace)
fig.update_layout(
    title=(
        "Character-count and line-width probes, values 50–100<br>"
        f"Joint PCA at {PROBE_ACTIVATION_SITE}"
    ),
    scene=pca_scene(residual_pca),
    height=PLOT_HEIGHT,
    margin=dict(l=0, r=0, b=0, t=90),
    legend=dict(orientation="h", y=1.02, x=0),
)
fig.show(renderer="notebook", config={"scrollZoom": True})

print("Residual-space joint PCA variance:", residual_pca.explained_variance_ratio_)
print(f"Layer: {LAYER}")

## Joint query/key geometry for every downstream attention head

For each head independently, character-count directions are mapped with `W_Q` and line-width directions with `W_K`. A fresh joint 3-D PCA is fit to that head's combined 102 embeddings, so each panel shows the geometry visible to that head in its own query/key feature space. Gray segments again pair equal values from 50 through 100.


In [ ]:
n_heads = cfg["model"]["n_heads"]
query_matrices = model.W_Q[ATTENTION_LAYER].detach().cpu().float().numpy()
key_matrices = model.W_K[ATTENTION_LAYER].detach().cpu().float().numpy()

head_joint_pcas = []
head_character_coordinates = []
head_width_coordinates = []

for head in range(n_heads):
    character_query_embeddings = (
        character_probe_vectors_50_100 @ query_matrices[head]
    )
    line_width_key_embeddings = line_width_probe_vectors @ key_matrices[head]
    head_pca, character_coordinates, width_coordinates = joint_pca_coordinates(
        character_query_embeddings,
        line_width_key_embeddings,
    )
    head_joint_pcas.append(head_pca)
    head_character_coordinates.append(character_coordinates)
    head_width_coordinates.append(width_coordinates)

n_columns = min(2, n_heads)
n_rows = int(np.ceil(n_heads / n_columns))
subplot_titles = [
    f"Layer {ATTENTION_LAYER}, head {head}: character × W_Q; width × W_K"
    for head in range(n_heads)
]
fig = make_subplots(
    rows=n_rows,
    cols=n_columns,
    specs=[[{"type": "scene"} for _ in range(n_columns)] for _ in range(n_rows)],
    subplot_titles=subplot_titles,
    vertical_spacing=0.08,
    horizontal_spacing=0.04,
)

for head in range(n_heads):
    row, column = divmod(head, n_columns)
    for trace in paired_probe_traces(
        head_character_coordinates[head],
        head_width_coordinates[head],
        character_name="Character-count × W_Q",
        width_name="Line-width × W_K",
        show_legend=head == 0,
    ):
        fig.add_trace(trace, row=row + 1, col=column + 1)
    scene_name = "scene" if head == 0 else f"scene{head + 1}"
    fig.update_layout(**{scene_name: pca_scene(head_joint_pcas[head])})

fig.update_layout(
    title=f"Attention layer {ATTENTION_LAYER}: joint query/key probe geometry",
    height=PLOT_HEIGHT * n_rows,
    margin=dict(l=0, r=0, b=0, t=110),
    legend=dict(orientation="h", y=1.02, x=0),
)
fig.show(renderer="notebook", config={"scrollZoom": True})

for head, pca in enumerate(head_joint_pcas):
    print(
        f"Layer {ATTENTION_LAYER}, head {head} joint PCA variance: "
        f"{pca.explained_variance_ratio_}"
    )
